# 03 · OANDA costs confirmed

One-shot audit of the locked `A_FX_OANDA_S3` table. Writes
`04_backtest/s3_fx_trend/artifacts/oanda_costs_confirmed.json`.


## 0. Imports & Config


In [ ]:
import json
import os
import sys
import warnings
from datetime import date

import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.fx_fetcher import G10_V1_PAIRS, fetch_fx_ohlcv
from data.ingestion.rates_fetcher import fetch_all_g10_policy_rates, rate_differential
from strategies.s3_fx_trend.costs import (
    COSTS,
    daily_swap_return,
    fx_pip_size,
    leg_cost_bps,
)

PROFILE = "A_FX_OANDA_S3"
cfg = COSTS[PROFILE]
ARTIFACT = os.path.join(ROOT, "04_backtest", "s3_fx_trend", "artifacts", "oanda_costs_confirmed.json")


## 1. Locked spreads → bps at current mid


In [ ]:
rows = []
for pair, pips in cfg["pair_spread_pips"].items():
    mid = float("nan")
    try:
        px = fetch_fx_ohlcv(pair, "2024-01-01", None, interval="1d", source="yfinance")
        if px is not None and not px.empty:
            mid = float(px["close"].iloc[-1])
    except Exception as exc:
        warnings.warn(f"{pair} mid fetch failed: {exc!r}")
    bps = leg_cost_bps(pair, mid) if mid == mid else float("nan")
    rows.append({
        "pair": pair,
        "spread_pips": pips,
        "pip_size": fx_pip_size(pair),
        "mid": mid,
        "leg_cost_bps_approx": bps,
        "slippage_pips": cfg["slippage_pips_per_leg"],
    })
tbl = pd.DataFrame(rows)
display(tbl)
assert set(G10_V1_PAIRS) <= set(cfg["pair_spread_pips"])
print("SEK/NOK absent from table (expected).")


## 2. Example swap PnL ($100k long EURUSD, 30 days)


In [ ]:
try:
    rates = fetch_all_g10_policy_rates(start="2020-01-01")
    # rates may be in percent — treat as decimal if abs mean > 1
    if not rates.empty and rates.abs().mean().mean() > 1:
        rates = rates / 100.0
    diff = rate_differential("EURUSD", rates)
    rd = float(diff.dropna().iloc[-1]) if not diff.dropna().empty else 0.0
except Exception as exc:
    warnings.warn(f"rates unavailable: {exc!r}")
    rd = 0.0
    rates = pd.DataFrame()

notional = 100_000.0
weight = 1.0  # unit weight; scale PnL by notional outside
accrual = 0.0
asof = pd.Timestamp("2024-06-03")  # Monday
for i in range(30):
    d = asof + pd.Timedelta(days=i)
    accrual += daily_swap_return(
        "EURUSD", weight, rd,
        financing_spread_bps_annual=cfg["swap"]["financing_spread_bps_annual"],
        wednesday_triple=cfg["swap"]["wednesday_triple_swap"],
        asof=d,
    )
print("rate_diff (base-quote)≈", rd)
print("30d swap return (unit weight)≈", accrual)
print("30d swap PnL on $100k≈", accrual * notional)


## 3. Write confirmation JSON


In [ ]:
payload = {
    "profile": PROFILE,
    "confirmed": True,
    "doc_date": str(date.today()),
    "pair_spread_pips": cfg["pair_spread_pips"],
    "slippage_pips_per_leg": cfg["slippage_pips_per_leg"],
    "swap": cfg["swap"],
    "conversion_markup_bps": cfg.get("conversion_markup_bps", 0.0),
    "example_eurusd_30d_swap_return": accrual,
    "note": "research lock — revisit if OANDA published schedule diverges",
}
os.makedirs(os.path.dirname(ARTIFACT), exist_ok=True)
with open(ARTIFACT, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
    f.write("\n")
print("wrote", ARTIFACT)
